# S3 (2024)
---
[[paper]](https://arxiv.org/pdf/2407.01953)<br>
S3 = Search, Select, and Serve

__S3__ — это системный фреймворк для оптимизации инференса Large Language Models (LLM), который радикально повышает пропускную способность (Throughput) при работе с длинными контекстами. Метод основан на динамическом предсказании и извлечении только тех частей KV cache, которые действительно необходимы для генерации текущего токена, вместо обработки всей истории контекста.

__Постановка задачи__<br>
При генерации текста в LLM основной проблемой является рост KV cache. Для каждого нового токена модель должна вычислить Attention по всем предыдущим токенам. В задачах с длинным контекстом (Long-context Generation) или RAG объем KV cache растет линейно, что приводит к дефициту памяти GPU и замедлению инференса из-за ограничений Memory Bandwidth. Задача S3 — минимизировать количество данных KV cache, участвующих в вычислениях, без потери качества генерации.

__Мотивация__<br>
Современные системы обслуживания (Serving Systems) тратят до 80% времени на чтение KV cache из памяти в вычислительные ядра. Анализ паттернов внимания показывает, что большинство Attention heads в трансформере являются разреженными: для предсказания следующего токена реально важны лишь 5-10% предыдущих токенов. Если мы научимся заранее определять эти "важные" токены, мы сможем сократить объем вычислений и передачи данных в несколько раз.

__Существующие подходы__<br>
До появления S3 использовались следующие методы оптимизации KV cache:
- PagedAttention (vLLM, 2023): оптимизирует управление памятью через фрагментацию, но все равно хранит и считывает полный KV cache для каждого запроса.
- H2O (2023): использует жадный алгоритм для удаления "неважных" токенов из кэша (Eviction). Проблема в том, что вытеснение происходит статически — однажды удаленный токен нельзя вернуть, даже если он понадобится для будущих генераций.
- StreamingLLM (2023): сохраняет только фиксированное окно последних токенов и первые токены (Attention Sinks). Метод неэффективен для задач, требующих доступа к середине длинного документа.

__Идея__<br>
Вместо того чтобы хранить меньше или удалять данные навсегда, авторы предложили концепцию динамического извлечения. S3 рассматривает KV cache как "базу данных", а процесс вычисления Attention — как поисковый запрос. Перед основным проходом модели запускается легковесный предиктор, который предсказывает, какие блоки контекста будут иметь наибольшие веса внимания. Это позволяет загружать из памяти GPU только релевантные фрагменты.

__Архитектура__<br>
Система состоит из трех ключевых компонентов:
1. Predictor: маленькая нейросеть (обычно двухслойный MLP или линейный слой), которая обучается предсказывать важность блоков токенов на основе текущего Hidden State.
2. KV Cache Manager: специализированный менеджер памяти, который организует KV cache в виде блоков и поддерживает быстрый доступ к произвольным индексам.
3. Sparse Attention Engine: кастомное ядро (Kernel), которое умеет эффективно вычислять Attention только по выбранному подмножеству индексов.

__Алгоритм обучения__<br>
Обучается только модуль Predictor, веса основной LLM остаются замороженными:
1. Сбор данных: Прогоняется набор текстов через целевую LLM, собираются реальные карты Attention (Ground Truth) для каждого слоя и каждой головы.
2. Формирование меток: Для каждого блока токенов (например, размером 16 или 64) вычисляется агрегированный вес внимания. Блоки с максимальными весами помечаются как целевые.
3. Обучение предиктора: Решается задача классификации или ранжирования. Предиктор учится по вектору текущего состояния (Query) предсказывать вероятности того, что конкретный блок KV (Key/Value) будет востребован.

__Алгоритм инференса__<br>
Процесс генерации каждого токена разделен на три фазы:
1. Search: Текущий Hidden State подается в Predictor. Он выдает список индексов блоков KV cache, которые, скорее всего, получат высокие веса внимания в текущем слое.
2. Select: Система фильтрует предсказанные блоки, применяя порог (Threshold) или выбирая Top-K. Это гарантирует, что объем передаваемых данных не превысит заданный лимит.
3. Serve: Выполняется стандартный проход трансформера, но в блоке Self-Attention участвуют только выбранные KV-пары. Остальные данные даже не считываются из основной видеопамяти в кэш L2/регистры GPU.

__Результаты__<br>
Метод тестировался на моделях Llama-2-7B (2023) и Mistral-7B (2023) с контекстами до 32k токенов:
- Throughput увеличился в 6.4 раза по сравнению с vLLM при сохранении идентичного уровня Perplexity.
- Использование памяти KV cache сократилось на 80% без значимой потери точности в задачах LongBench (падение метрик составило менее 1%).
- S3 показал новизну в том, что первым реализовал "предсказательный" Sparse Attention на уровне системы обслуживания, а не только как теоретическую конструкцию архитектуры модели.

## 📝 Критический анализ

```markdown
# S3 (2024)
---
[[paper]](https://arxiv.org/pdf/2407.01953)<br>
S3 = Search, Select, and Serve

__S3__ — системный фреймворк для оптимизации инференса Large Language Models (LLM), повышающий пропускную способность при работе с длинными контекстами. Метод динамически предсказывает и извлекает только необходимые части KV cache для генерации текущего токена, избегая обработки всей истории контекста.

__Постановка задачи__<br>
При генерации текста в LLM основная проблема — рост KV cache. Для каждого нового токена модель должна вычислить Attention по всем предыдущим токенам, что приводит к дефициту памяти GPU и замедлению инференса. S3 минимизирует объем данных KV cache, участвующих в вычислениях, без потери качества генерации.

__Мотивация__<br>
Современные системы обслуживания тратят до 80% времени на чтение KV cache. Анализ показывает, что для предсказания следующего токена важны лишь 5-10% предыдущих токенов. Если заранее определять эти "важные" токены, можно сократить объем вычислений и передачи данных.

__Существующие подходы__<br>
- PagedAttention (vLLM, 2023): оптимизирует память через фрагментацию, но хранит полный KV cache.
- H2O (2023): удаляет "неважные" токены, но статически.
- StreamingLLM (2023): сохраняет только фиксированное окно последних и первых токенов.

__Идея__<br>
S3 предлагает динамическое извлечение. KV cache рассматривается как "база данных", а вычисление Attention — как поисковый запрос. Легковесный предиктор предсказывает, какие блоки контекста будут иметь наибольшие веса внимания, загружая из памяти GPU только релевантные фрагменты.

<img src="img/img.png" width=500>

__Архитектура__<br>
1. Predictor: маленькая нейросеть, предсказывающая важность блоков токенов.
2. KV Cache Manager: организует KV cache в виде блоков для быстрого доступа.
3. Sparse Attention Engine: вычисляет Attention только по выбранным индексам.

__Алгоритм обучения__<br>
Обучается только Predictor, веса LLM остаются замороженными:
1. Сбор данных: Прогон текстов через LLM, сбор карт Attention.
2. Формирование меток: Вычисление агрегированных весов внимания для блоков токенов.
3. Обучение предиктора: Предсказание вероятностей востребованности блоков KV.

__Алгоритм инференса__<br>
1. Search: Predictor выдает индексы блоков KV cache с высокими весами внимания.
2. Select: Фильтрация предсказанных блоков, применение порога или выбор Top-K.
3. Serve: Проход трансформера с участием только выбранных KV-пар.

__Результаты__<br>
Метод тестировался на моделях Llama-2-7B (2023) и Mistral-7B (2023) с контекстами до 32k токенов:
- Throughput увеличился в 6.4 раза по сравнению с vLLM при сохранении уровня Perplexity.
- Использование памяти KV cache сократилось на 80% без значимой потери точности в задачах LongBench (падение метрик менее 1%).
- S3 первым реализовал "предсказательный" Sparse Attention на уровне системы обслуживания.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dummy data to simulate KV cache and hidden states
KV_cache = torch.rand(1000, 64)  # Simulated KV cache with 1000 blocks of size 64
hidden_state = torch.rand(64)    # Current hidden state of size 64

# Predictor: A simple MLP to predict important KV blocks
class Predictor(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Predictor, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))  # Output probabilities for each block
        return x

# Initialize the predictor
predictor = Predictor(input_size=64, hidden_size=32, output_size=1000)

# Search Phase: Use the predictor to find important blocks
block_importance = predictor(hidden_state)
# Select top-K important blocks
top_k_blocks = torch.topk(block_importance, k=10).indices

# Select Phase: Filter the KV cache to only include important blocks
selected_KV_cache = KV_cache[top_k_blocks]

# Serve Phase: Perform attention only on selected KV cache
# Dummy attention mechanism for illustration
def sparse_attention(query, keys, values):
    # Simple dot-product attention for demonstration
    scores = torch.matmul(query, keys.T)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, values)
    return output

# Simulate query, keys, and values
query = hidden_state
keys = selected_KV_cache
values = selected_KV_cache  # In practice, values might be different

# Perform sparse attention
output = sparse_attention(query, keys, values)

# Output represents the result of the attention mechanism
print("Attention Output:", output)

# Explanation:
# 1. Predictor is a small neural network that predicts the importance of each block in the KV cache.
# 2. During the Search phase, the predictor outputs probabilities indicating the importance of each block.
# 3. In the Select phase, we choose the top-K important blocks based on the predictor's output.
# 4. The Serve phase involves performing attention only on the selected blocks, reducing memory and computation.
# 5. This approach dynamically selects relevant parts of the KV cache, optimizing memory usage and throughput.
```

This code demonstrates the S3 method's core idea: dynamically predicting and selecting relevant parts of the KV cache for efficient attention computation. The Predictor is a lightweight MLP that estimates the importance of each block, allowing the system to focus only on the most relevant data, thus optimizing memory and computational resources.